In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import date, datetime, timedelta
from pyspark.sql.window import Window

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_base_motivos")

# display(df_bronze_motivos)

### Tratamento da tabela `base_motivos`

Esta etapa padroniza as colunas `criticidade` e `categoria`, corrigindo os problemas encontrados no diagnóstico da camada bronze.

---

#### 1. Padronização da `criticidade`

**Problema:** o mesmo nível de criticidade aparecia escrito de formas diferentes (`Alta` e `alta`). Em agrupamentos e filtros, essas variações seriam tratadas como níveis distintos.

**Tratamento:** os espaços nas pontas são removidos (`trim`) e o texto é convertido para minúsculo (`lower`).

| Antes | Depois |
|---|---|
| `Alta` / `alta` | `alta` |
| `Media` | `media` |
| `baixa` | `baixa` |

**Resultado:** a coluna passa a ter apenas três valores: `alta`, `media` e `baixa`.

---

#### 2. Preenchimento e padronização da `categoria`

**Problema:** três motivos estavam sem categoria (ids 12, 14 e 16). O valor vazio pode chegar como `null` ou como texto vazio (`""`), dependendo da leitura do CSV, por isso os dois casos são tratados.

**Tratamento:**
- Categorias nulas ou vazias recebem o valor `Outro`.
- As demais têm os espaços nas pontas removidos.
- Todo o resultado é convertido para minúsculo, seguindo o mesmo padrão da `criticidade`.

| id_motivo | nome_motivo | Antes | Depois |
|---|---|---|---|
| 12 | Contratação de cartão adicional | *(vazio)* | `outro` |
| 14 | Transferência de agência | *(vazio)* | `outro` |
| 16 | Cancelamento de conta | *(vazio)* | `outro` |

**Resultado:** a coluna passa a ter os valores `cartao`, `financeiro`, `beneficios`, `atendimento`, `cadastral` e `outro`.

**Ponto de atenção:** o motivo 12 (*Contratação de cartão adicional*) provavelmente pertence à categoria `cartao`. A atribuição correta dos três motivos deve ser confirmada com a área de negócio. Até lá, eles permanecem como `outro`.

In [0]:
#Padroniza em minusculo
df_silver_motivos = df_bronze_motivos.withColumn("criticidade", F.lower(F.trim(F.col("criticidade"))))

#Transforma os null em categoria em outro
df_silver_motivos = df_silver_motivos.withColumn(
    "categoria",
    F.lower(
        F.when(
            F.col("categoria").isNull() | (F.trim(F.col("categoria")) == ""),
            F.lit("Outro")
        ).otherwise(F.trim(F.col("categoria")))
    )
)

display(df_silver_motivos)

id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime
11,Bloqueio de cartão,cartao,alta,2026-10-19T19:37:37.013Z
7,Compra não autorizada,cartao,alta,2026-10-19T19:37:37.013Z
13,Dúvidas gerais sobre programa de pontos,beneficios,baixa,2026-10-19T19:37:37.013Z
14,Transferência de agência,outro,media,2026-10-19T19:37:37.013Z
9,Problema com aplicativo,atendimento,media,2026-10-19T19:37:37.013Z
8,Consulta de contrato,financeiro,media,2026-10-19T19:37:37.013Z
15,Renegociação de dívida,financeiro,alta,2026-10-19T19:37:37.013Z
16,Cancelamento de conta,outro,alta,2026-10-19T19:37:37.013Z
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",cadastral,baixa,2026-10-19T19:37:37.013Z
3,Contestação de fatura,financeiro,alta,2026-10-19T19:37:37.013Z


In [0]:
df_silver_motivos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_motivos")


print(f"Tabela {silver_schema}.tb_chamados criada/atualizada com sucesso!")
print(f"Bronze (antes do dedup): {df_bronze_motivos.count():,} linhas")
print(f"Silver (após dedup e tratamento): {df_silver_motivos.count():,} linhas")